# DemandPulse
## Forecast Monitoring, Drift Detection & Retraining Strategy

**Notebook 06 — Production-Style Forecast Observability**

This notebook continues the DemandPulse forecasting project and simulates a production monitoring layer around the previously trained forecasting artifacts.

It answers five operational questions:

1. Is incoming data structurally valid and complete?
2. Are model inputs or demand distributions drifting away from the training reference period?
3. Is forecast accuracy or probabilistic calibration degrading over time?
4. Are failures concentrated in particular stores, products, categories, demand levels, or horizons?
5. Is there enough persistent evidence to recommend retraining — without retraining merely because drift exists?

The notebook **does not automatically retrain or deploy a model**. It produces monitoring evidence and a retraining recommendation only.


## 1. Environment

The first executable cell installs only the packages used here. Warnings are not globally suppressed.


In [ ]:
%pip install -q "datasets>=3.0" "pandas>=2.0" "numpy>=1.24" "pyarrow>=14" "matplotlib>=3.7" "scikit-learn>=1.3" "lightgbm>=4.0" "scipy>=1.10" "tqdm>=4.66"


In [ ]:
import gc
import hashlib
import json
import random
import sys
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import datasets
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from datasets import load_dataset
from scipy.stats import ks_2samp
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm.auto import tqdm

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("LightGBM:", lgb.__version__)


## 2. Monitoring configuration

Thresholds below are **monitoring policy assumptions**, not universal statistical laws. They are centralized so a future production owner can tune them using business risk, historical incident rates, and model governance requirements.

The reference period is the final part of the training split. The official `eval` period acts as chronological "recent production" data. Monitoring is evaluated by target date to simulate successive production periods.


In [ ]:
DATASET_ID = "Dingdong-Inc/FreshRetailNet-50K"
SERIES_SAMPLE_PERCENT = 30
SERIES_KEYS = ["store_id", "product_id"]
STREAM_BATCH_SIZE = 50_000

REFERENCE_DAYS = 28
MAX_LAG_DAYS = 28
CONTEXT_DAYS = REFERENCE_DAYS + MAX_LAG_DAYS + 7
MONITORING_FREQUENCY = "D"
MIN_SEGMENT_OBSERVATIONS = 20
MIN_NEW_OBSERVATIONS_FOR_RETRAIN = 5_000
PERSISTENCE_PERIODS = 2
NOMINAL_INTERVAL_COVERAGE = 0.80

# Data-quality thresholds: fraction of observations unless stated otherwise.
DQ_THRESHOLDS = {
    "missing_cell_rate": (0.01, 0.05),
    "max_column_missing_rate": (0.02, 0.10),
    "duplicate_rate": (0.001, 0.01),
    "invalid_value_rate": (0.001, 0.01),
    "new_store_rate": (0.01, 0.05),
    "new_product_rate": (0.02, 0.10),
}

# Drift thresholds. PSI is often interpreted around 0.10 / 0.25; here those are configurable alerts.
PSI_YELLOW, PSI_RED = 0.10, 0.25
KS_YELLOW, KS_RED = 0.10, 0.20
TARGET_MEAN_SHIFT_YELLOW, TARGET_MEAN_SHIFT_RED = 0.15, 0.30
TARGET_ZERO_RATE_DELTA_YELLOW, TARGET_ZERO_RATE_DELTA_RED = 0.05, 0.10
TARGET_VARIANCE_LOG_RATIO_YELLOW, TARGET_VARIANCE_LOG_RATIO_RED = 0.25, 0.50

# Performance degradation versus chronological backtest reference.
PERF_DEGRADATION_YELLOW, PERF_DEGRADATION_RED = 0.10, 0.20
ABS_BIAS_YELLOW, ABS_BIAS_RED = 0.08, 0.15
COVERAGE_GAP_YELLOW, COVERAGE_GAP_RED = 0.05, 0.10

REPORTS_DIR = Path("reports")
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = Path("models")
QUANTILE_MODELS_DIR = MODELS_DIR / "quantile"
HIER_MODELS_DIR = MODELS_DIR / "hierarchical"
for directory in [REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, QUANTILE_MODELS_DIR, HIER_MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

OUTPUT_PATHS = {
    "data_quality": TABLES_DIR / "data_quality_monitoring.csv",
    "feature_drift": TABLES_DIR / "feature_drift.csv",
    "performance": TABLES_DIR / "forecast_performance_monitoring.csv",
    "calibration": TABLES_DIR / "calibration_monitoring.csv",
    "retraining": TABLES_DIR / "retraining_decisions.csv",
}
print(json.dumps({k: str(v) for k, v in OUTPUT_PATHS.items()}, indent=2))


## 3. Prior DemandPulse artifact and model registry

Production monitoring must know **what model version it is monitoring**. This notebook fingerprints model files with SHA-256 and checks that the probabilistic prediction artifact produced by Notebook 02 exists.

The compact Notebook 02 evaluation-prediction artifact is the primary monitoring feed because it contains actual demand, P10/P50/P90, conformal intervals, target dates, and forecast horizons on the same observations.

Notebook 01 point, Notebook 04 hierarchical, Notebook 03 causal, and Notebook 05 inventory artifacts are registered when available, but this notebook does not depend on Python variables from any earlier kernel.


In [ ]:
CORE_REQUIRED_ARTIFACTS = {
    "probabilistic_eval_predictions": TABLES_DIR / "probabilistic_eval_predictions.parquet",
    "probabilistic_backtest_results": TABLES_DIR / "probabilistic_backtest_results.csv",
}
OPTIONAL_PROJECT_ARTIFACTS = {
    "point_model": MODELS_DIR / "demandpulse_lgbm.txt",
    "probabilistic_final_metrics": TABLES_DIR / "probabilistic_final_metrics.csv",
    "hierarchical_forecasts": TABLES_DIR / "hierarchical_forecasts.csv",
    "hierarchy_level_metrics": TABLES_DIR / "hierarchy_level_metrics.csv",
    "causal_ate": TABLES_DIR / "ate_estimates.csv",
    "inventory_policy_comparison": TABLES_DIR / "inventory_policy_comparison.csv",
}
QUANTILE_MODEL_PATHS = [
    QUANTILE_MODELS_DIR / f"h{h:02d}_q{q:02d}.txt"
    for h in range(1, 8)
    for q in (10, 50, 90)
]


def file_sha256(path: Path, chunk_size: int = 1 << 20) -> Optional[str]:
    """Return a stable model/artifact fingerprint without reading the whole file into memory."""
    if not path.exists():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        while True:
            chunk = fh.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

registry_rows = []
for name, path in {**CORE_REQUIRED_ARTIFACTS, **OPTIONAL_PROJECT_ARTIFACTS}.items():
    registry_rows.append({
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "sha256": file_sha256(path),
    })
for path in QUANTILE_MODEL_PATHS:
    registry_rows.append({
        "artifact": path.stem,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "sha256": file_sha256(path),
    })
model_registry = pd.DataFrame(registry_rows)
display(model_registry)
model_registry.to_csv(TABLES_DIR / "monitoring_model_registry.csv", index=False)

missing_core = [name for name, path in CORE_REQUIRED_ARTIFACTS.items() if not path.exists()]
missing_quantile_models = [str(path) for path in QUANTILE_MODEL_PATHS if not path.exists()]
if missing_core:
    raise FileNotFoundError(
        "Notebook 06 requires executed Notebook 02 monitoring artifacts. Missing: " + ", ".join(missing_core)
    )
if missing_quantile_models:
    raise FileNotFoundError(
        "Notebook 06 expects the 21 saved Notebook 02 quantile model files. Missing examples: "
        + ", ".join(missing_quantile_models[:3])
    )

# Verify model files are readable by the installed LightGBM version without retaining 21 boosters in RAM.
for path in tqdm(QUANTILE_MODEL_PATHS, desc="Validating quantile model files"):
    booster = lgb.Booster(model_file=str(path))
    if booster.num_trees() <= 0:
        raise RuntimeError(f"Invalid or empty LightGBM model: {path}")
    del booster
gc.collect()
print("Core monitoring artifacts and 21 quantile model files validated.")


## 4. Remote production/reference data access

The raw FreshRetailNet-50K source remains read-only and remote. We reuse the exact deterministic complete-series sampling rule from earlier DemandPulse notebooks:

`SHA256("store_id-product_id") → first 8 bytes → modulo 10,000 → sample bucket`.

To remain memory conscious, the training stream is scanned once for metadata and entity history. A second pass materializes only the tail required for the reference window plus lag context. The smaller `eval` split is materialized for recent monitoring.


In [ ]:
EXPECTED_SCHEMA = [
    "city_id", "store_id", "management_group_id", "first_category_id", "second_category_id",
    "third_category_id", "product_id", "dt", "sale_amount", "hours_sale",
    "stock_hour6_22_cnt", "hours_stock_status", "discount", "holiday_flag", "activity_flag",
    "precpt", "avg_temperature", "avg_humidity", "avg_wind_level",
]
MATERIALIZED_COLUMNS = [
    "city_id", "store_id", "management_group_id", "first_category_id", "second_category_id",
    "third_category_id", "product_id", "dt", "sale_amount", "stock_hour6_22_cnt", "discount",
    "holiday_flag", "activity_flag", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level",
]


def series_selected(store_id: object, product_id: object, percent: float, cache: Dict[Tuple, bool]) -> bool:
    """Notebook-01-compatible deterministic complete-series selection."""
    if not 0 < percent <= 100:
        raise ValueError("percent must be in (0, 100].")
    if percent >= 100:
        return True
    key = (store_id, product_id)
    if key not in cache:
        token = f"{store_id}-{product_id}".encode("utf-8")
        digest = hashlib.sha256(token).digest()
        bucket = int.from_bytes(digest[:8], byteorder="big", signed=False) % 10_000
        cache[key] = bucket < int(round(percent * 100))
    return cache[key]


def selected_batch_mask(batch: Mapping[str, Sequence], cache: Dict[Tuple, bool]) -> np.ndarray:
    return np.fromiter(
        (
            series_selected(s, p, SERIES_SAMPLE_PERCENT, cache)
            for s, p in zip(batch["store_id"], batch["product_id"])
        ),
        dtype=bool,
        count=len(batch["store_id"]),
    )

# Determinism sanity checks.
_cache_a, _cache_b = {}, {}
assert series_selected("1", "2", 30, _cache_a) == series_selected("1", "2", 30, _cache_b)
assert series_selected("1", "2", 100, {})

train_stream = load_dataset(DATASET_ID, split="train", streaming=True)
eval_stream = load_dataset(DATASET_ID, split="eval", streaming=True)

train_schema = list(train_stream.features.keys())
eval_schema = list(eval_stream.features.keys())
missing_train = sorted(set(EXPECTED_SCHEMA) - set(train_schema))
missing_eval = sorted(set(EXPECTED_SCHEMA) - set(eval_schema))
if missing_train or missing_eval:
    raise AssertionError(f"Required schema fields missing. train={missing_train}; eval={missing_eval}")

print("Train stream type:", type(train_stream).__name__)
print("Eval stream type:", type(eval_stream).__name__)
print("Schema fields:", len(train_schema))


In [ ]:
def scan_train_metadata(stream) -> Dict[str, object]:
    """Scan the selected training series to establish chronology and historical entity universe."""
    cache: Dict[Tuple, bool] = {}
    min_dt = None
    max_dt = None
    stores, products, categories, series = set(), set(), set(), set()
    rows_processed = rows_selected = 0

    for batch in tqdm(stream.batch(batch_size=STREAM_BATCH_SIZE), desc="Scanning train metadata", unit="batch"):
        rows_processed += len(batch["store_id"])
        mask = selected_batch_mask(batch, cache)
        rows_selected += int(mask.sum())
        if not mask.any():
            continue
        dates = pd.to_datetime(np.asarray(batch["dt"], dtype=object)[mask], errors="raise")
        batch_min, batch_max = dates.min(), dates.max()
        min_dt = batch_min if min_dt is None else min(min_dt, batch_min)
        max_dt = batch_max if max_dt is None else max(max_dt, batch_max)
        selected_store = np.asarray(batch["store_id"], dtype=object)[mask]
        selected_product = np.asarray(batch["product_id"], dtype=object)[mask]
        selected_cat = np.asarray(batch["third_category_id"], dtype=object)[mask]
        stores.update(selected_store.tolist())
        products.update(selected_product.tolist())
        categories.update(selected_cat.tolist())
        series.update(zip(selected_store.tolist(), selected_product.tolist()))

    if max_dt is None:
        raise RuntimeError("No sampled training rows were selected.")
    return {
        "rows_processed": rows_processed,
        "rows_selected": rows_selected,
        "min_dt": pd.Timestamp(min_dt),
        "max_dt": pd.Timestamp(max_dt),
        "stores": stores,
        "products": products,
        "categories": categories,
        "series": series,
    }

train_meta = scan_train_metadata(train_stream)
print({k: v if not isinstance(v, set) else len(v) for k, v in train_meta.items()})


In [ ]:
def materialize_selected_stream(
    stream,
    split_name: str,
    min_date: Optional[pd.Timestamp] = None,
    max_date: Optional[pd.Timestamp] = None,
) -> pd.DataFrame:
    """Batch-materialize selected complete series and an optional chronological window."""
    cache: Dict[Tuple, bool] = {}
    frames: List[pd.DataFrame] = []
    processed = selected = retained = 0

    for batch in tqdm(stream.batch(batch_size=STREAM_BATCH_SIZE), desc=f"Materializing {split_name}", unit="batch"):
        processed += len(batch["store_id"])
        mask = selected_batch_mask(batch, cache)
        selected += int(mask.sum())
        if not mask.any():
            continue
        frame = pd.DataFrame({c: batch[c] for c in MATERIALIZED_COLUMNS})
        frame = frame.loc[mask].copy()
        frame["dt"] = pd.to_datetime(frame["dt"], errors="raise")
        if min_date is not None:
            frame = frame.loc[frame["dt"] >= pd.Timestamp(min_date)]
        if max_date is not None:
            frame = frame.loc[frame["dt"] <= pd.Timestamp(max_date)]
        if not frame.empty:
            retained += len(frame)
            frames.append(frame)

    if not frames:
        raise RuntimeError(f"No rows retained while materializing {split_name}.")
    out = pd.concat(frames, ignore_index=True, copy=False)
    print(f"{split_name}: processed={processed:,}; sampled={selected:,}; retained={retained:,}")
    return out

train_max = train_meta["max_dt"]
context_start = train_max - pd.Timedelta(days=CONTEXT_DAYS - 1)
reference_start = train_max - pd.Timedelta(days=REFERENCE_DAYS - 1)

# Reload the streams because the previous training iterator was consumed.
train_context_stream = load_dataset(DATASET_ID, split="train", streaming=True)
eval_recent_stream = load_dataset(DATASET_ID, split="eval", streaming=True)

train_context_df = materialize_selected_stream(
    train_context_stream, "train reference context", min_date=context_start, max_date=train_max
)
eval_df = materialize_selected_stream(eval_recent_stream, "eval recent")

if not (train_context_df["dt"].max() < eval_df["dt"].min()):
    raise RuntimeError(
        f"Chronology violation: train max={train_context_df['dt'].max()} eval min={eval_df['dt'].min()}. "
        "Monitoring performance would risk temporal leakage."
    )

print("Reference context shape:", train_context_df.shape)
print("Recent eval shape:", eval_df.shape)
print("Reference window:", reference_start.date(), "to", train_max.date())
print("Recent window:", eval_df["dt"].min().date(), "to", eval_df["dt"].max().date())
print("Memory MB:", round((train_context_df.memory_usage(deep=True).sum() + eval_df.memory_usage(deep=True).sum()) / 1024**2, 2))


## 5. Data-quality monitoring

Data-quality failures are separated from model drift. A model should **not** be retrained to compensate for broken inputs.

Checks include:
- expected schema and feature-type changes,
- missingness,
- duplicate `(store_id, product_id, dt)` records,
- negative demand / stock fields,
- invalid binary flags or discount ranges,
- stores/products not observed in the full sampled training history.

A RED data-quality state blocks a retraining recommendation until the data issue is resolved.


In [ ]:
STATUS_ORDER = {"GREEN": 0, "YELLOW": 1, "RED": 2, "NOT_ASSESSED": -1}


def threshold_status(value: float, yellow: float, red: float, absolute: bool = False) -> str:
    if not np.isfinite(value):
        return "NOT_ASSESSED"
    x = abs(value) if absolute else value
    if x >= red:
        return "RED"
    if x >= yellow:
        return "YELLOW"
    return "GREEN"


def max_status(statuses: Iterable[str]) -> str:
    valid = [s for s in statuses if s in STATUS_ORDER and s != "NOT_ASSESSED"]
    return max(valid, key=lambda s: STATUS_ORDER[s]) if valid else "NOT_ASSESSED"


def schema_monitor_rows(train_stream, eval_stream) -> List[Dict[str, object]]:
    train_features = train_stream.features
    eval_features = eval_stream.features
    train_cols, eval_cols = set(train_features.keys()), set(eval_features.keys())
    required_missing = sorted(set(EXPECTED_SCHEMA) - eval_cols)
    extras = sorted(eval_cols - set(EXPECTED_SCHEMA))
    common = sorted(train_cols & eval_cols)
    type_changes = [c for c in common if str(train_features[c]) != str(eval_features[c])]
    return [
        {"monitoring_period": "SCHEMA", "check": "required_schema_missing", "value": len(required_missing),
         "status": "RED" if required_missing else "GREEN", "details": str(required_missing)},
        {"monitoring_period": "SCHEMA", "check": "unexpected_schema_fields", "value": len(extras),
         "status": "YELLOW" if extras else "GREEN", "details": str(extras)},
        {"monitoring_period": "SCHEMA", "check": "schema_type_changes", "value": len(type_changes),
         "status": "RED" if type_changes else "GREEN", "details": str(type_changes)},
    ]


def data_quality_rows(frame: pd.DataFrame, period_label: str) -> List[Dict[str, object]]:
    n = len(frame)
    if n == 0:
        return [{"monitoring_period": period_label, "check": "empty_batch", "value": 1.0, "status": "RED", "details": "No rows"}]

    core_cols = [c for c in MATERIALIZED_COLUMNS if c in frame.columns]
    missing_cell_rate = frame[core_cols].isna().to_numpy().mean()
    col_missing = frame[core_cols].isna().mean()
    duplicate_rate = frame.duplicated(SERIES_KEYS + ["dt"], keep=False).mean()

    invalid_target = frame["sale_amount"].isna() | (frame["sale_amount"] < 0)
    invalid_stock = frame["stock_hour6_22_cnt"].notna() & (frame["stock_hour6_22_cnt"] < 0)
    invalid_discount = frame["discount"].notna() & ~frame["discount"].between(0, 1)
    invalid_flags = (~frame["holiday_flag"].isin([0, 1]) & frame["holiday_flag"].notna()) | (
        ~frame["activity_flag"].isin([0, 1]) & frame["activity_flag"].notna()
    )
    invalid_rate = (invalid_target | invalid_stock | invalid_discount | invalid_flags).mean()

    new_store_rate = (~frame["store_id"].isin(train_meta["stores"])).mean()
    new_product_rate = (~frame["product_id"].isin(train_meta["products"])).mean()

    checks = {
        "missing_cell_rate": float(missing_cell_rate),
        "max_column_missing_rate": float(col_missing.max()),
        "duplicate_rate": float(duplicate_rate),
        "invalid_value_rate": float(invalid_rate),
        "new_store_rate": float(new_store_rate),
        "new_product_rate": float(new_product_rate),
    }
    rows = []
    for check, value in checks.items():
        yellow, red = DQ_THRESHOLDS[check]
        details = ""
        if check == "max_column_missing_rate":
            details = f"worst_column={col_missing.idxmax()}"
        rows.append({
            "monitoring_period": period_label,
            "check": check,
            "value": value,
            "yellow_threshold": yellow,
            "red_threshold": red,
            "status": threshold_status(value, yellow, red),
            "details": details,
        })
    return rows

# Schema rows plus chronological recent batches.
quality_records = schema_monitor_rows(
    load_dataset(DATASET_ID, split="train", streaming=True),
    load_dataset(DATASET_ID, split="eval", streaming=True),
)
eval_df["monitoring_period"] = eval_df["dt"].dt.floor(MONITORING_FREQUENCY)
for period, frame in eval_df.groupby("monitoring_period", sort=True):
    quality_records.extend(data_quality_rows(frame, str(pd.Timestamp(period).date())))

data_quality_monitoring = pd.DataFrame(quality_records)
data_quality_monitoring.to_csv(OUTPUT_PATHS["data_quality"], index=False)
display(data_quality_monitoring)

# Explicitly display duplicates before any refusal; no duplicates are silently deleted.
duplicates = eval_df.loc[eval_df.duplicated(SERIES_KEYS + ["dt"], keep=False)].sort_values(SERIES_KEYS + ["dt"])
print("Duplicate recent records:", len(duplicates))
if not duplicates.empty:
    display(duplicates.head(20))


## 6. Leakage-safe monitoring features

Feature drift is measured on variables that can be available to the forecasting system, not on future target information.

Historical demand and stockout variables are shifted by series. Same-day `sale_amount` is reserved for **target drift and post-outcome performance evaluation**. Same-day stockout counts are not treated as forecast features.


In [ ]:
def build_monitoring_features(context: pd.DataFrame) -> pd.DataFrame:
    """Create leakage-safe historical and known exogenous monitoring features."""
    out = context.sort_values(SERIES_KEYS + ["dt"]).copy()
    grouped = out.groupby(SERIES_KEYS, sort=False, observed=True)

    out["lag_1"] = grouped["sale_amount"].shift(1)
    out["lag_7"] = grouped["sale_amount"].shift(7)
    out["lag_28"] = grouped["sale_amount"].shift(28)
    out["rolling_mean_7"] = grouped["sale_amount"].transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
    out["rolling_mean_28"] = grouped["sale_amount"].transform(lambda s: s.shift(1).rolling(28, min_periods=7).mean())
    out["rolling_std_7"] = grouped["sale_amount"].transform(lambda s: s.shift(1).rolling(7, min_periods=3).std())
    out["stockout_lag_1"] = grouped["stock_hour6_22_cnt"].shift(1)
    out["stockout_lag_7"] = grouped["stock_hour6_22_cnt"].shift(7)

    out["day_of_week"] = out["dt"].dt.dayofweek.astype("int8")
    out["month"] = out["dt"].dt.month.astype("int8")
    out["is_weekend"] = out["day_of_week"].isin([5, 6]).astype("int8")
    out["discount_pct"] = 1.0 - out["discount"]
    return out

combined_context = pd.concat(
    [train_context_df.assign(source="reference_context"), eval_df.drop(columns=["monitoring_period"]).assign(source="recent")],
    ignore_index=True,
)
feature_context = build_monitoring_features(combined_context)
reference_features = feature_context.loc[
    feature_context["dt"].between(reference_start, train_max, inclusive="both")
].copy()
recent_features = feature_context.loc[feature_context["source"].eq("recent")].copy()
recent_features["monitoring_period"] = recent_features["dt"].dt.floor(MONITORING_FREQUENCY)

DRIFT_FEATURES = [
    "lag_1", "lag_7", "lag_28", "rolling_mean_7", "rolling_mean_28", "rolling_std_7",
    "stockout_lag_1", "stockout_lag_7", "day_of_week", "month", "is_weekend",
    "discount_pct", "holiday_flag", "activity_flag", "precpt", "avg_temperature",
    "avg_humidity", "avg_wind_level",
]

for forbidden in ["sale_amount", "stock_hour6_22_cnt", "hours_sale", "hours_stock_status"]:
    assert forbidden not in DRIFT_FEATURES, f"Leakage-sensitive feature found in DRIFT_FEATURES: {forbidden}"

print("Reference feature rows:", len(reference_features))
print("Recent feature rows:", len(recent_features))
print("Drift features:", DRIFT_FEATURES)


## 7. Feature drift: PSI + KS

Two complementary diagnostics are used:

- **Population Stability Index (PSI)** compares binned population proportions. Reference quantiles define bins for continuous variables so recent data cannot influence the binning scheme.
- **Kolmogorov–Smirnov statistic (KS)** measures the maximum difference between empirical CDFs for numeric variables. The statistic is emphasized over the p-value because very large samples can make tiny changes statistically significant.

PSI/KS thresholds are operational alert rules. They do not by themselves prove model degradation.


In [ ]:
def population_stability_index(reference: pd.Series, recent: pd.Series, bins: int = 10, eps: float = 1e-6) -> float:
    """PSI using reference-derived quantile bins for continuous features and categories otherwise."""
    ref = reference.copy()
    cur = recent.copy()
    if pd.api.types.is_numeric_dtype(ref) and ref.dropna().nunique() > 10:
        clean_ref = ref.dropna().astype(float)
        quantiles = np.unique(np.quantile(clean_ref, np.linspace(0, 1, bins + 1)))
        if len(quantiles) < 3:
            return 0.0
        edges = np.concatenate(([-np.inf], quantiles[1:-1], [np.inf]))
        ref_bins = pd.cut(ref.astype(float), bins=edges, include_lowest=True).astype(str).fillna("__MISSING__")
        cur_bins = pd.cut(cur.astype(float), bins=edges, include_lowest=True).astype(str).fillna("__MISSING__")
    else:
        ref_bins = ref.astype("object").where(ref.notna(), "__MISSING__").astype(str)
        cur_bins = cur.astype("object").where(cur.notna(), "__MISSING__").astype(str)

    categories = sorted(set(ref_bins.unique()) | set(cur_bins.unique()))
    ref_dist = ref_bins.value_counts(normalize=True).reindex(categories, fill_value=0.0).clip(lower=eps)
    cur_dist = cur_bins.value_counts(normalize=True).reindex(categories, fill_value=0.0).clip(lower=eps)
    return float(((cur_dist - ref_dist) * np.log(cur_dist / ref_dist)).sum())


def ks_diagnostic(reference: pd.Series, recent: pd.Series) -> Tuple[float, float]:
    ref = pd.to_numeric(reference, errors="coerce").dropna().to_numpy(dtype=float)
    cur = pd.to_numeric(recent, errors="coerce").dropna().to_numpy(dtype=float)
    if len(ref) < 2 or len(cur) < 2:
        return np.nan, np.nan
    result = ks_2samp(ref, cur, alternative="two-sided", method="auto")
    return float(result.statistic), float(result.pvalue)


def drift_status(psi: float, ks_stat: float) -> str:
    statuses = [threshold_status(psi, PSI_YELLOW, PSI_RED)]
    if np.isfinite(ks_stat):
        statuses.append(threshold_status(ks_stat, KS_YELLOW, KS_RED))
    return max_status(statuses)

feature_drift_records = []
for period, recent in recent_features.groupby("monitoring_period", sort=True):
    for feature in DRIFT_FEATURES:
        psi = population_stability_index(reference_features[feature], recent[feature])
        ks_stat, ks_p = ks_diagnostic(reference_features[feature], recent[feature])
        feature_drift_records.append({
            "monitoring_period": pd.Timestamp(period),
            "drift_type": "feature",
            "feature": feature,
            "reference_observations": int(reference_features[feature].notna().sum()),
            "recent_observations": int(recent[feature].notna().sum()),
            "psi": psi,
            "ks_statistic": ks_stat,
            "ks_pvalue": ks_p,
            "status": drift_status(psi, ks_stat),
        })

feature_drift = pd.DataFrame(feature_drift_records)
feature_drift.to_csv(OUTPUT_PATHS["feature_drift"], index=False)
display(feature_drift.sort_values(["monitoring_period", "psi"], ascending=[True, False]).head(30))


## 8. Target drift and intermittency

Target drift is monitored separately from feature drift because `sale_amount` becomes known only after the demand outcome is observed.

We track:
- mean demand,
- variance,
- zero-sales rate,
- average per-series zero-sales ratio (an intermittency indicator).

These changes can indicate concept shift or a changing business mix, but they do not identify a causal reason.


In [ ]:
def target_summary(frame: pd.DataFrame) -> Dict[str, float]:
    y = frame["sale_amount"].astype(float)
    series_zero_ratio = frame.assign(zero=(y == 0).astype(float)).groupby(SERIES_KEYS, observed=True)["zero"].mean()
    return {
        "mean_demand": float(y.mean()),
        "demand_variance": float(y.var(ddof=1)),
        "zero_sales_rate": float((y == 0).mean()),
        "mean_series_zero_sales_ratio": float(series_zero_ratio.mean()),
        "observations": int(len(frame)),
    }

reference_target = target_summary(reference_features)
target_drift_records = []
for period, recent in recent_features.groupby("monitoring_period", sort=True):
    cur = target_summary(recent)
    mean_shift = abs(cur["mean_demand"] - reference_target["mean_demand"]) / max(abs(reference_target["mean_demand"]), 1e-12)
    variance_log_ratio = abs(np.log((cur["demand_variance"] + 1e-12) / (reference_target["demand_variance"] + 1e-12)))
    zero_delta = abs(cur["zero_sales_rate"] - reference_target["zero_sales_rate"])
    intermittency_delta = abs(cur["mean_series_zero_sales_ratio"] - reference_target["mean_series_zero_sales_ratio"])

    statuses = [
        threshold_status(mean_shift, TARGET_MEAN_SHIFT_YELLOW, TARGET_MEAN_SHIFT_RED),
        threshold_status(variance_log_ratio, TARGET_VARIANCE_LOG_RATIO_YELLOW, TARGET_VARIANCE_LOG_RATIO_RED),
        threshold_status(zero_delta, TARGET_ZERO_RATE_DELTA_YELLOW, TARGET_ZERO_RATE_DELTA_RED),
        threshold_status(intermittency_delta, TARGET_ZERO_RATE_DELTA_YELLOW, TARGET_ZERO_RATE_DELTA_RED),
    ]
    target_drift_records.append({
        "monitoring_period": pd.Timestamp(period),
        **{f"reference_{k}": v for k, v in reference_target.items()},
        **{f"recent_{k}": v for k, v in cur.items()},
        "mean_demand_relative_shift": mean_shift,
        "variance_abs_log_ratio": variance_log_ratio,
        "zero_sales_rate_abs_delta": zero_delta,
        "intermittency_abs_delta": intermittency_delta,
        "status": max_status(statuses),
    })

target_drift = pd.DataFrame(target_drift_records)
target_drift.to_csv(TABLES_DIR / "target_drift.csv", index=False)
display(target_drift)


## 9. Monitoring prediction feed

The Notebook 02 Parquet artifact is treated like a production prediction log: it contains the prediction timestamp relationship (`origin_date`, `target_date`, `horizon`) and the later observed target.

All metrics use a single explicit common mask requiring actual, P10, P50 and P90. Conformal coverage is computed on the same rows when conformal bounds are available.


In [ ]:
PREDICTION_PATH = CORE_REQUIRED_ARTIFACTS["probabilistic_eval_predictions"]
predictions = pd.read_parquet(PREDICTION_PATH)
for c in ["origin_date", "target_date"]:
    predictions[c] = pd.to_datetime(predictions[c], errors="raise")

required_prediction_columns = [
    *SERIES_KEYS, "target_date", "horizon", "target", "p_p10", "p_p50", "p_p90",
    "conformal_lower", "conformal_upper",
]
missing_prediction_cols = sorted(set(required_prediction_columns) - set(predictions.columns))
if missing_prediction_cols:
    raise AssertionError(f"Prediction artifact missing required columns: {missing_prediction_cols}")

# Join production dimensions from the real eval feed.
eval_dimensions = eval_df[[
    *SERIES_KEYS, "dt", "city_id", "third_category_id", "discount", "holiday_flag", "activity_flag"
]].rename(columns={"dt": "target_date"})
monitor = predictions.merge(eval_dimensions, on=SERIES_KEYS + ["target_date"], how="left", validate="many_to_one")

# Prediction target must match the same official eval outcome after the join.
eval_target = eval_df[[*SERIES_KEYS, "dt", "sale_amount"]].rename(columns={"dt": "target_date", "sale_amount": "eval_target"})
monitor = monitor.merge(eval_target, on=SERIES_KEYS + ["target_date"], how="left", validate="many_to_one")
comparison = monitor[["target", "eval_target"]].dropna()
if len(comparison) and not np.allclose(comparison["target"], comparison["eval_target"], rtol=1e-7, atol=1e-9):
    raise AssertionError("Notebook 02 target values do not align with the current official eval feed.")

common_evaluation_mask = monitor[["target", "p_p10", "p_p50", "p_p90"]].notna().all(axis=1)
print("Prediction rows:", len(monitor))
print("Common evaluation rows:", int(common_evaluation_mask.sum()))
print("Prediction coverage %:", round(100 * common_evaluation_mask.mean(), 3))
monitor = monitor.loc[common_evaluation_mask].copy()
monitor["monitoring_period"] = monitor["target_date"].dt.floor(MONITORING_FREQUENCY)
assert (monitor["target_date"] > monitor["origin_date"]).all(), "Target dates must follow forecast origins."
assert monitor["horizon"].between(1, 7).all(), "Notebook 06 expects the fixed 7-day Notebook 02 evaluation horizons."

# Training-derived demand-level labels, never eval-outcome-derived thresholds.
series_reference = reference_features.groupby(SERIES_KEYS, observed=True)["sale_amount"].mean().rename("reference_mean_demand").reset_index()
q_low, q_high = series_reference["reference_mean_demand"].quantile([1/3, 2/3]).tolist()
def demand_level_from_reference(x: float) -> str:
    if not np.isfinite(x): return "Unknown/Cold Start"
    if x <= q_low: return "Low"
    if x <= q_high: return "Medium"
    return "High"
series_reference["demand_level"] = series_reference["reference_mean_demand"].map(demand_level_from_reference)
monitor = monitor.merge(series_reference, on=SERIES_KEYS, how="left")
monitor["demand_level"] = monitor["demand_level"].fillna("Unknown/Cold Start")

display(monitor.head())


## 10. Forecast performance monitoring

Definitions match the earlier DemandPulse notebooks:

- **MAE**: mean absolute error.
- **RMSE**: square root of mean squared error.
- **WAPE**: `sum(|y - ŷ|) / sum(|y|)`, guarded against a zero denominator.
- **Bias**: `sum(ŷ - y) / sum(|y|)`. Positive means systematic **overforecasting**; negative means **underforecasting**.
- **Pinball loss**: quantile loss at P10/P50/P90.
- **Coverage**: fraction of actuals within P10–P90.
- **Interval width**: P90 − P10.

Metrics are generated by date and by store, product, category, demand level and forecast horizon. Small segment rows below the configured minimum are excluded from segment reporting, not from overall metrics.


In [ ]:
def safe_wape(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    y, p = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y).sum()
    return float(np.abs(y - p).sum() / denom) if denom > 0 else np.nan


def safe_bias(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    y, p = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y).sum()
    return float((p - y).sum() / denom) if denom > 0 else np.nan


def pinball_loss(y_true: Sequence[float], y_pred: Sequence[float], quantile: float) -> float:
    y, p = np.asarray(y_true, float), np.asarray(y_pred, float)
    error = y - p
    return float(np.mean(np.where(error >= 0, quantile * error, (quantile - 1) * error)))


def performance_metrics(frame: pd.DataFrame) -> Dict[str, float]:
    y = frame["target"].to_numpy(float)
    p10 = frame["p_p10"].to_numpy(float)
    p50 = frame["p_p50"].to_numpy(float)
    p90 = frame["p_p90"].to_numpy(float)
    if len(y) == 0:
        return {k: np.nan for k in ["mae", "rmse", "wape", "bias", "p10_pinball", "p50_pinball", "p90_pinball", "mean_pinball", "coverage", "interval_width"]} | {"observations": 0}
    losses = [pinball_loss(y, p10, .10), pinball_loss(y, p50, .50), pinball_loss(y, p90, .90)]
    return {
        "mae": float(mean_absolute_error(y, p50)),
        "rmse": float(mean_squared_error(y, p50) ** 0.5),
        "wape": safe_wape(y, p50),
        "bias": safe_bias(y, p50),
        "p10_pinball": losses[0],
        "p50_pinball": losses[1],
        "p90_pinball": losses[2],
        "mean_pinball": float(np.mean(losses)),
        "coverage": float(np.mean((p10 <= y) & (y <= p90))),
        "interval_width": float(np.mean(p90 - p10)),
        "observations": int(len(y)),
    }


def performance_by_scope(frame: pd.DataFrame, scope: str, segment_column: Optional[str] = None) -> List[Dict[str, object]]:
    rows = []
    group_cols = ["monitoring_period"] + ([segment_column] if segment_column else [])
    for keys, group in frame.groupby(group_cols, observed=True, sort=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        if segment_column and len(group) < MIN_SEGMENT_OBSERVATIONS:
            continue
        period = pd.Timestamp(keys[0])
        segment_value = keys[1] if segment_column else "ALL"
        rows.append({
            "model": "LightGBM Quantile P50/P10-P90",
            "scope": scope,
            "segment": str(segment_value),
            "monitoring_period": period,
            **performance_metrics(group),
        })
    return rows

performance_records = []
performance_records += performance_by_scope(monitor, "overall")
for scope, column in [
    ("store", "store_id"),
    ("product", "product_id"),
    ("category", "third_category_id"),
    ("demand_level", "demand_level"),
    ("forecast_horizon", "horizon"),
]:
    performance_records += performance_by_scope(monitor, scope, column)

forecast_performance_monitoring = pd.DataFrame(performance_records)
forecast_performance_monitoring.to_csv(OUTPUT_PATHS["performance"], index=False)

overall_performance = forecast_performance_monitoring.loc[forecast_performance_monitoring["scope"].eq("overall")].copy()
display(overall_performance)
print("Segment monitoring rows:", len(forecast_performance_monitoring))


### Optional hierarchical point-model comparison

When Notebook 04's `hierarchical_forecasts.csv` is available, this section adds daily SKU-level MinT-style point-forecast metrics. It is kept separate from probabilistic calibration because Notebook 04 does not produce P10/P90 intervals.


In [ ]:
hier_path = OPTIONAL_PROJECT_ARTIFACTS["hierarchical_forecasts"]
if hier_path.exists():
    hier = pd.read_csv(hier_path)
    hier["target_date"] = pd.to_datetime(hier["target_date"], errors="raise")
    required_hier = {"level", "method", "target_date", "forecast", "actual", "horizon"}
    if required_hier.issubset(hier.columns):
        mint = hier.loc[hier["level"].eq("SKU") & hier["method"].eq("MinT-Style")].copy()
        mint["monitoring_period"] = mint["target_date"].dt.floor(MONITORING_FREQUENCY)
        hier_rows = []
        for period, group in mint.groupby("monitoring_period", sort=True):
            y = group["actual"].to_numpy(float)
            p = group["forecast"].to_numpy(float)
            hier_rows.append({
                "model": "Hierarchical MinT-Style SKU",
                "scope": "overall",
                "segment": "ALL",
                "monitoring_period": pd.Timestamp(period),
                "mae": float(mean_absolute_error(y, p)),
                "rmse": float(mean_squared_error(y, p) ** 0.5),
                "wape": safe_wape(y, p),
                "bias": safe_bias(y, p),
                "p10_pinball": np.nan, "p50_pinball": np.nan, "p90_pinball": np.nan,
                "mean_pinball": np.nan, "coverage": np.nan, "interval_width": np.nan,
                "observations": len(group),
            })
        hierarchical_monitoring = pd.DataFrame(hier_rows)
        forecast_performance_monitoring = pd.concat([forecast_performance_monitoring, hierarchical_monitoring], ignore_index=True)
        forecast_performance_monitoring.to_csv(OUTPUT_PATHS["performance"], index=False)
        display(hierarchical_monitoring)
    else:
        print("Hierarchical forecast artifact exists but does not expose expected columns; skipping optional monitor.")
else:
    print("Notebook 04 hierarchy prediction artifact not found; probabilistic monitoring continues normally.")


## 11. Calibration monitoring

Probabilistic forecasts can degrade even when point MAE looks stable. We therefore monitor:

- empirical P10/P50/P90 probabilities,
- raw P10–P90 interval coverage,
- conformal interval coverage,
- raw and conformal interval width.

A calibration alert is based on the absolute difference from nominal 80% coverage. Conformal intervals are monitored separately from the raw quantile interval.


In [ ]:
def calibration_metrics(frame: pd.DataFrame) -> Dict[str, float]:
    y = frame["target"].to_numpy(float)
    p10, p50, p90 = (frame[c].to_numpy(float) for c in ["p_p10", "p_p50", "p_p90"])
    raw_coverage = float(np.mean((p10 <= y) & (y <= p90)))
    raw_width = float(np.mean(p90 - p10))
    conformal_mask = frame[["conformal_lower", "conformal_upper"]].notna().all(axis=1).to_numpy()
    if conformal_mask.any():
        lo = frame.loc[conformal_mask, "conformal_lower"].to_numpy(float)
        hi = frame.loc[conformal_mask, "conformal_upper"].to_numpy(float)
        yc = frame.loc[conformal_mask, "target"].to_numpy(float)
        conformal_coverage = float(np.mean((lo <= yc) & (yc <= hi)))
        conformal_width = float(np.mean(hi - lo))
    else:
        conformal_coverage = conformal_width = np.nan
    return {
        "empirical_p10": float(np.mean(y <= p10)),
        "empirical_p50": float(np.mean(y <= p50)),
        "empirical_p90": float(np.mean(y <= p90)),
        "raw_coverage": raw_coverage,
        "raw_coverage_gap": raw_coverage - NOMINAL_INTERVAL_COVERAGE,
        "raw_interval_width": raw_width,
        "conformal_coverage": conformal_coverage,
        "conformal_coverage_gap": conformal_coverage - NOMINAL_INTERVAL_COVERAGE if np.isfinite(conformal_coverage) else np.nan,
        "conformal_interval_width": conformal_width,
        "observations": int(len(frame)),
    }

calibration_records = []
for scope, column in [("overall", None), ("forecast_horizon", "horizon"), ("category", "third_category_id"), ("demand_level", "demand_level")]:
    group_cols = ["monitoring_period"] + ([column] if column else [])
    for keys, group in monitor.groupby(group_cols, observed=True, sort=True):
        if not isinstance(keys, tuple): keys = (keys,)
        if column and len(group) < MIN_SEGMENT_OBSERVATIONS: continue
        metrics = calibration_metrics(group)
        raw_status = threshold_status(abs(metrics["raw_coverage_gap"]), COVERAGE_GAP_YELLOW, COVERAGE_GAP_RED)
        conformal_status = threshold_status(abs(metrics["conformal_coverage_gap"]), COVERAGE_GAP_YELLOW, COVERAGE_GAP_RED) if np.isfinite(metrics["conformal_coverage_gap"]) else "NOT_ASSESSED"
        calibration_records.append({
            "scope": scope,
            "segment": str(keys[1]) if column else "ALL",
            "monitoring_period": pd.Timestamp(keys[0]),
            **metrics,
            "raw_status": raw_status,
            "conformal_status": conformal_status,
            "status": max_status([raw_status, conformal_status]),
        })

calibration_monitoring = pd.DataFrame(calibration_records)
calibration_monitoring.to_csv(OUTPUT_PATHS["calibration"], index=False)
display(calibration_monitoring.loc[calibration_monitoring["scope"].eq("overall")])


## 12. Chronological performance reference and degradation detection

Current performance should not be judged against the official eval period itself. The reference comes from Notebook 02's rolling-origin **training-period backtests**.

Where possible, the benchmark is horizon-specific. This avoids penalizing a seven-day-ahead forecast simply because it is naturally harder than a one-day-ahead forecast.


In [ ]:
backtest = pd.read_csv(CORE_REQUIRED_ARTIFACTS["probabilistic_backtest_results"])
method_mask = backtest["method"].astype(str).str.contains("LightGBM Quantile", regex=False)
backtest_model = backtest.loc[method_mask].copy()
if backtest_model.empty:
    raise RuntimeError("No LightGBM quantile rows found in probabilistic backtest results.")

required_bt_metrics = ["mae", "wape", "bias", "mean_pinball", "coverage", "mean_width"]
missing_bt_metrics = [c for c in required_bt_metrics if c not in backtest_model.columns]
if missing_bt_metrics:
    raise AssertionError(f"Backtest artifact lacks required monitoring metrics: {missing_bt_metrics}")

performance_reference_by_horizon = backtest_model.groupby("horizon", observed=True)[required_bt_metrics].mean().reset_index()
performance_reference_by_horizon = performance_reference_by_horizon.rename(columns={c: f"reference_{c}" for c in required_bt_metrics})
display(performance_reference_by_horizon)

# Every official fixed-origin eval date maps to one horizon; compare with the matching backtest horizon.
overall_with_horizon = monitor.groupby("monitoring_period", observed=True)["horizon"].agg(lambda x: int(pd.Series(x).mode().iloc[0])).rename("horizon").reset_index()
current_overall = overall_performance.merge(overall_with_horizon, on="monitoring_period", how="left", validate="one_to_one")
current_overall = current_overall.merge(performance_reference_by_horizon, on="horizon", how="left", validate="many_to_one")

for metric in ["mae", "wape", "mean_pinball"]:
    denom = current_overall[f"reference_{metric}"].abs().replace(0, np.nan)
    current_overall[f"{metric}_relative_degradation"] = (current_overall[metric] - current_overall[f"reference_{metric}"]) / denom

current_overall["performance_status"] = [
    max_status([
        threshold_status(max(row.wape_relative_degradation, 0) if np.isfinite(row.wape_relative_degradation) else np.nan, PERF_DEGRADATION_YELLOW, PERF_DEGRADATION_RED),
        threshold_status(max(row.mean_pinball_relative_degradation, 0) if np.isfinite(row.mean_pinball_relative_degradation) else np.nan, PERF_DEGRADATION_YELLOW, PERF_DEGRADATION_RED),
        threshold_status(row.bias, ABS_BIAS_YELLOW, ABS_BIAS_RED, absolute=True),
    ])
    for row in current_overall.itertuples(index=False)
]
current_overall["bias_direction"] = np.select(
    [current_overall["bias"] < -ABS_BIAS_YELLOW, current_overall["bias"] > ABS_BIAS_YELLOW],
    ["SYSTEMATIC_UNDERFORECAST", "SYSTEMATIC_OVERFORECAST"],
    default="BALANCED_WITHIN_THRESHOLD",
)
display(current_overall[[
    "monitoring_period", "horizon", "mae", "wape", "bias", "mean_pinball", "coverage",
    "wape_relative_degradation", "mean_pinball_relative_degradation", "performance_status", "bias_direction"
]])


## 13. Retraining decision policy

**Drift is evidence to investigate, not an automatic retraining trigger.**

A retraining recommendation requires all of the following:

1. enough new labeled observations,
2. persistent forecast-performance or calibration degradation for multiple monitoring periods,
3. persistent feature or target drift for multiple periods,
4. no RED data-quality incident that should be fixed first.

This design avoids replacing a sound model because of one noisy period or because the input distribution changed without harming forecast performance.


In [ ]:
def bad_streak(statuses: Sequence[str]) -> List[int]:
    streak, out = 0, []
    for status in statuses:
        streak = streak + 1 if status in {"YELLOW", "RED"} else 0
        out.append(streak)
    return out

periods = sorted(monitor["monitoring_period"].unique())
period_summary_rows = []

quality_daily = data_quality_monitoring.loc[data_quality_monitoring["monitoring_period"].ne("SCHEMA")].copy()
quality_daily["monitoring_period"] = pd.to_datetime(quality_daily["monitoring_period"])
quality_status = quality_daily.groupby("monitoring_period")["status"].agg(max_status)
feature_status = feature_drift.groupby("monitoring_period")["status"].agg(max_status)
target_status = target_drift.set_index("monitoring_period")["status"]
calib_overall = calibration_monitoring.loc[calibration_monitoring["scope"].eq("overall")].set_index("monitoring_period")
perf_overall = current_overall.set_index("monitoring_period")
new_obs = monitor.groupby("monitoring_period").size()

for period in periods:
    p = pd.Timestamp(period)
    period_summary_rows.append({
        "monitoring_period": p,
        "new_labeled_observations": int(new_obs.get(p, 0)),
        "data_quality_status": quality_status.get(p, "NOT_ASSESSED"),
        "feature_drift_status": feature_status.get(p, "NOT_ASSESSED"),
        "target_drift_status": target_status.get(p, "NOT_ASSESSED"),
        "performance_status": perf_overall.loc[p, "performance_status"] if p in perf_overall.index else "NOT_ASSESSED",
        "calibration_status": calib_overall.loc[p, "status"] if p in calib_overall.index else "NOT_ASSESSED",
        "wape": perf_overall.loc[p, "wape"] if p in perf_overall.index else np.nan,
        "bias": perf_overall.loc[p, "bias"] if p in perf_overall.index else np.nan,
        "coverage": calib_overall.loc[p, "raw_coverage"] if p in calib_overall.index else np.nan,
    })

retraining_decisions = pd.DataFrame(period_summary_rows).sort_values("monitoring_period").reset_index(drop=True)
retraining_decisions["drift_status"] = [max_status([f, t]) for f, t in zip(retraining_decisions["feature_drift_status"], retraining_decisions["target_drift_status"])]
retraining_decisions["degradation_status"] = [max_status([p, c]) for p, c in zip(retraining_decisions["performance_status"], retraining_decisions["calibration_status"])]
retraining_decisions["drift_bad_streak"] = bad_streak(retraining_decisions["drift_status"].tolist())
retraining_decisions["degradation_bad_streak"] = bad_streak(retraining_decisions["degradation_status"].tolist())
retraining_decisions["enough_new_observations"] = retraining_decisions["new_labeled_observations"] >= MIN_NEW_OBSERVATIONS_FOR_RETRAIN

choices, reasons, overall_status = [], [], []
for row in retraining_decisions.itertuples(index=False):
    statuses = [row.data_quality_status, row.drift_status, row.degradation_status]
    status = max_status(statuses)

    if row.data_quality_status == "RED":
        decision = "FIX_DATA_FIRST"
        reason = "Severe data-quality failure; retraining could learn corrupted inputs."
    elif not row.enough_new_observations:
        decision = "INSUFFICIENT_DATA"
        reason = "Alert evidence exists, but too few new labeled observations for a retraining decision."
    elif row.degradation_bad_streak >= PERSISTENCE_PERIODS and row.drift_bad_streak >= PERSISTENCE_PERIODS:
        decision = "RETRAIN_RECOMMENDED"
        reason = "Persistent model/calibration degradation coincides with persistent drift and enough new labels."
        status = "RED"
    elif row.degradation_status in {"YELLOW", "RED"}:
        decision = "MONITOR"
        reason = "Performance/calibration degraded, but persistence and/or corroborating drift is insufficient."
        status = max_status([status, "YELLOW"])
    elif row.drift_status in {"YELLOW", "RED"}:
        decision = "NO_RETRAIN"
        reason = "Drift detected without material forecast degradation; investigate but do not retrain automatically."
        status = max_status([status, "YELLOW"])
    else:
        decision = "NO_RETRAIN"
        reason = "No material persistent degradation signal."
        status = "GREEN" if status != "RED" else status

    choices.append(decision)
    reasons.append(reason)
    overall_status.append(status)

retraining_decisions["decision"] = choices
retraining_decisions["reason"] = reasons
retraining_decisions["monitoring_status"] = overall_status
retraining_decisions.to_csv(OUTPUT_PATHS["retraining"], index=False)
display(retraining_decisions)


## 14. Segment hot spots

Overall metrics can hide local failures. This section identifies segments with unusually high WAPE, absolute bias, or calibration error, subject to the minimum-observation rule.


In [ ]:
segment_perf = forecast_performance_monitoring.loc[
    forecast_performance_monitoring["model"].eq("LightGBM Quantile P50/P10-P90")
    & ~forecast_performance_monitoring["scope"].eq("overall")
].copy()
segment_perf["abs_bias"] = segment_perf["bias"].abs()
segment_perf["coverage_gap"] = (segment_perf["coverage"] - NOMINAL_INTERVAL_COVERAGE).abs()

worst_wape = segment_perf.sort_values("wape", ascending=False).head(15)
worst_bias = segment_perf.sort_values("abs_bias", ascending=False).head(15)
worst_calibration = segment_perf.sort_values("coverage_gap", ascending=False).head(15)

print("Highest WAPE monitored segments")
display(worst_wape[["monitoring_period", "scope", "segment", "observations", "wape", "bias", "coverage"]])
print("Largest absolute bias monitored segments")
display(worst_bias[["monitoring_period", "scope", "segment", "observations", "wape", "bias", "coverage"]])
print("Largest calibration gaps")
display(worst_calibration[["monitoring_period", "scope", "segment", "observations", "wape", "bias", "coverage"]])


## 15. Monitoring visualizations

The plots focus on alert interpretation rather than decoration.


In [ ]:
# Plot 1: top feature PSI trajectories.
top_features = feature_drift.groupby("feature")["psi"].max().nlargest(6).index.tolist()
fig, ax = plt.subplots(figsize=(10, 5))
for feature in top_features:
    sub = feature_drift.loc[feature_drift["feature"].eq(feature)].sort_values("monitoring_period")
    ax.plot(sub["monitoring_period"], sub["psi"], marker="o", label=feature)
ax.axhline(PSI_YELLOW, linestyle="--", linewidth=1, label="PSI yellow threshold")
ax.axhline(PSI_RED, linestyle=":", linewidth=1, label="PSI red threshold")
ax.set_title("Feature drift: highest PSI trajectories")
ax.set_xlabel("Monitoring period")
ax.set_ylabel("PSI")
ax.tick_params(axis="x", rotation=45)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "monitoring_feature_drift.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 2: WAPE and backtest reference by horizon.
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(current_overall["monitoring_period"], current_overall["wape"], marker="o", label="Recent WAPE")
ax.plot(current_overall["monitoring_period"], current_overall["reference_wape"], marker="x", linestyle="--", label="Backtest reference WAPE")
ax.set_title("Forecast WAPE vs chronological backtest reference")
ax.set_xlabel("Monitoring period")
ax.set_ylabel("WAPE")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "monitoring_wape.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 3: interval coverage.
cal_plot = calibration_monitoring.loc[calibration_monitoring["scope"].eq("overall")].sort_values("monitoring_period")
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cal_plot["monitoring_period"], cal_plot["raw_coverage"], marker="o", label="Raw P10-P90 coverage")
ax.plot(cal_plot["monitoring_period"], cal_plot["conformal_coverage"], marker="s", label="Conformal coverage")
ax.axhline(NOMINAL_INTERVAL_COVERAGE, linestyle="--", linewidth=1, label="Nominal 80%")
ax.set_ylim(0, 1)
ax.set_title("Prediction interval calibration over time")
ax.set_xlabel("Monitoring period")
ax.set_ylabel("Empirical coverage")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "monitoring_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 4: bias, where negative means underforecasting and positive means overforecasting.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(current_overall["monitoring_period"], current_overall["bias"], marker="o")
ax.axhline(0, linewidth=1)
ax.axhline(ABS_BIAS_YELLOW, linestyle="--", linewidth=1)
ax.axhline(-ABS_BIAS_YELLOW, linestyle="--", linewidth=1)
ax.set_title("Forecast bias over time")
ax.set_xlabel("Monitoring period")
ax.set_ylabel("Bias (+ overforecast, − underforecast)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "monitoring_bias.png", dpi=150, bbox_inches="tight")
plt.show()


## 16. Scientific monitoring findings

The following findings are generated from executed monitoring outputs. They describe the current simulation period only; they are not hard-coded claims.


In [ ]:
def pct(x: float) -> str:
    return "NA" if not np.isfinite(x) else f"{100*x:.2f}%"

latest = retraining_decisions.iloc[-1]
worst_feature = feature_drift.sort_values("psi", ascending=False).iloc[0]
worst_perf = current_overall.sort_values("wape_relative_degradation", ascending=False).iloc[0]
worst_cov = cal_plot.assign(gap=(cal_plot["raw_coverage"] - NOMINAL_INTERVAL_COVERAGE).abs()).sort_values("gap", ascending=False).iloc[0]
status_counts = retraining_decisions["monitoring_status"].value_counts().to_dict()
retrain_count = int((retraining_decisions["decision"] == "RETRAIN_RECOMMENDED").sum())

print("# Findings")
print(f"- Monitoring periods evaluated: {len(retraining_decisions)}; status counts: {status_counts}.")
print(f"- Largest observed feature PSI: {worst_feature['feature']} = {worst_feature['psi']:.4f} on {pd.Timestamp(worst_feature['monitoring_period']).date()} ({worst_feature['status']}).")
print(f"- Worst WAPE relative degradation vs matched rolling-backtest reference: {pct(worst_perf['wape_relative_degradation'])} on {pd.Timestamp(worst_perf['monitoring_period']).date()}.")
print(f"- The most extreme raw interval coverage gap occurred on {pd.Timestamp(worst_cov['monitoring_period']).date()}: coverage={pct(worst_cov['raw_coverage'])} vs nominal={pct(NOMINAL_INTERVAL_COVERAGE)}.")
print(f"- Latest bias={latest['bias']:.4f}; positive indicates aggregate overforecasting and negative indicates underforecasting.")
print(f"- Retraining was recommended in {retrain_count} monitoring period(s). Drift alone never triggers retraining in this policy.")
print(f"- Latest production-style status: {latest['monitoring_status']} / decision={latest['decision']} — {latest['reason']}")

if not worst_wape.empty:
    row = worst_wape.iloc[0]
    print(f"- Highest monitored segment WAPE: scope={row['scope']}, segment={row['segment']}, WAPE={row['wape']:.4f}, observations={int(row['observations'])}.")


## 17. Production monitoring strategy

A real DemandPulse deployment should separate **detection**, **diagnosis**, **retraining**, and **deployment**.

### Data and schema controls
- Validate schema, nullability, key uniqueness and numeric domains at ingestion.
- Quarantine RED-quality batches instead of letting them silently enter feature generation.
- Register new stores/products explicitly; cold-start traffic should be routed to an architecture that supports it.

### Drift monitoring
- Maintain a versioned reference dataset tied to each deployed model version.
- Compute PSI/KS for model inputs on a regular cadence and target drift only after actual demand is available.
- Treat statistical drift as an investigation signal; do not equate it with degraded forecasting.

### Performance and calibration
- Join forecasts to actuals by immutable prediction IDs / target timestamps.
- Monitor WAPE and Bias alongside MAE because aggregate inventory exposure depends on scale and direction.
- Monitor quantile pinball loss, empirical quantile probabilities, interval coverage and width. High coverage obtained only by very wide intervals is not automatically desirable.
- Keep store/product/category/horizon slices so aggregate averages cannot hide concentrated failures.

### Retraining policy
- Require enough new labeled data.
- Require persistence across multiple periods rather than one noisy interval.
- Prefer a joint signal: model/calibration degradation **plus** persistent feature/target drift.
- If data quality is RED, fix the pipeline before retraining.
- Retraining should create a **challenger**, not overwrite the champion automatically.

### Safe promotion of a retrained model
1. reproduce the original rolling-origin tests,
2. evaluate the challenger on a new untouched chronological holdout,
3. verify point accuracy, bias and probabilistic calibration,
4. validate hierarchy/inventory downstream compatibility,
5. run shadow or canary traffic,
6. promote only if predefined acceptance criteria pass,
7. preserve a rollback artifact and model/data lineage.

This notebook intentionally stops at the **retraining recommendation**. Automated retraining, registry promotion and deployment orchestration require a separate governed MLOps workflow.


## 18. Limitations

- The official `eval` split is a short, fixed-origin historical period, so this is a **production-style simulation**, not months of live telemetry.
- PSI, KS and alert thresholds are configurable governance rules; they should be calibrated to historical false-alert and incident costs.
- KS p-values can become tiny with large samples even when the practical distribution change is small; the KS statistic is therefore emphasized.
- Recent demand actuals arrive after the forecast is made, so performance and target-drift alerts are necessarily delayed in real production.
- Reference-window choice affects drift estimates. Seasonal businesses may need season-matched references rather than one fixed 28-day window.
- New-product detection is relative to the sampled historical entity universe when `SERIES_SAMPLE_PERCENT < 100`.
- Segment metrics can be noisy for low-volume entities; this notebook enforces a minimum row count but does not solve all small-sample uncertainty.
- A detected association between drift and model degradation does not identify the causal reason for the degradation.
- This notebook recommends retraining but does not fit, register, deploy or roll back a new model.


## 19. Artifact and quality audit


In [ ]:
# Save final outputs once more after all optional enrichments.
data_quality_monitoring.to_csv(OUTPUT_PATHS["data_quality"], index=False)
feature_drift.to_csv(OUTPUT_PATHS["feature_drift"], index=False)
forecast_performance_monitoring.to_csv(OUTPUT_PATHS["performance"], index=False)
calibration_monitoring.to_csv(OUTPUT_PATHS["calibration"], index=False)
retraining_decisions.to_csv(OUTPUT_PATHS["retraining"], index=False)

artifact_audit = pd.DataFrame([
    {
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    }
    for name, path in OUTPUT_PATHS.items()
])
display(artifact_audit)
assert artifact_audit["exists"].all(), "One or more mandatory monitoring outputs are missing."
assert (artifact_audit["size_bytes"] > 0).all(), "One or more mandatory monitoring outputs are empty."

# Final logical safety checks.
assert set(retraining_decisions["monitoring_status"]).issubset({"GREEN", "YELLOW", "RED", "NOT_ASSESSED"})
assert set(retraining_decisions["decision"]).issubset({"NO_RETRAIN", "MONITOR", "RETRAIN_RECOMMENDED", "FIX_DATA_FIRST", "INSUFFICIENT_DATA"})
assert not ((retraining_decisions["decision"] == "RETRAIN_RECOMMENDED") & (retraining_decisions["drift_bad_streak"] < PERSISTENCE_PERIODS)).any()
assert not ((retraining_decisions["decision"] == "RETRAIN_RECOMMENDED") & (retraining_decisions["degradation_bad_streak"] < PERSISTENCE_PERIODS)).any()
assert not ((retraining_decisions["decision"] == "RETRAIN_RECOMMENDED") & ~retraining_decisions["enough_new_observations"]).any()
assert not ((retraining_decisions["decision"] == "RETRAIN_RECOMMENDED") & retraining_decisions["data_quality_status"].eq("RED")).any()
assert (monitor["target_date"] > monitor["origin_date"]).all()
assert (train_context_df["dt"].max() < eval_df["dt"].min())
assert common_evaluation_mask.any()
assert len(QUANTILE_MODEL_PATHS) == 21
assert all(path.exists() for path in QUANTILE_MODEL_PATHS)

print("Quality audit passed.")
print("Notebook 06 complete.")
print("\nCompleted:")
print("✓ Data-quality monitoring")
print("✓ Feature drift (PSI / KS)")
print("✓ Target drift and intermittency")
print("✓ Point and probabilistic performance monitoring")
print("✓ Calibration monitoring")
print("✓ Segment diagnostics")
print("✓ GREEN / YELLOW / RED status")
print("✓ Persistence-aware retraining decisions")
print("\nNext production phase:")
print("Governed model registry, automated retraining pipeline, shadow deployment and rollback strategy.")
